### Inference on one tile
#### Main idea: build datacube, fill it for all the pixel for a certain window size (eg 8) time step by time step, then normalize --> rinse and repeat (33 separate data cubes in this example, which will be overwritten)

In [ ]:
import os
import time
import torch
import numpy as np
import rasterio
from rasterio.windows import Window
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import joblib
from multikernel_model import AlbasUNet

In [ ]:
# --- Config ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  
print(f"Using device: {device}")

# Optimize CUDA operations
torch.backends.cudnn.benchmark = True
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

# Paths
base_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net'
model_path = os.path.join(base_path, 'Multikernel_Model/best_model_2.pth')
dir_path = os.path.join(base_path, 'tiles/raw_data')
output_dir = os.path.join(base_path, 'Multikernel_Model/predictions')
scaler_path = os.path.join(base_path, 'Multikernel_Model/minmax_scaler.joblib')

In [ ]:
# Features and parameters
features_sep = ["BLU", "GRN", "RED", "NIR", "SW1", "SW2"]
band_indices = [0, 1, 2, 3, 4, 5]
n_features = len(band_indices)
window_size = 8
batch_size = 131072  # Increased batch size for better GPU utilization

In [ ]:
def load_and_preprocess_data(file_paths, band_indices, shape, dummy_value=-9999):
    """Load and preprocess data using normal sequential processing"""
    data_cube = np.zeros(shape, dtype=np.float32)
    
    #shape is (height, width, n_features, window_size)
    #idt is time index
    # rasterio format is: (bands, height, width)
    # data cube format is (height, width, bands)
    # file_paths depends indirectly on window size

    #file_paths is a list of file paths
    for idt, path in enumerate(tqdm(file_paths, desc="Loading files")):
        if os.path.exists(path):
            with rasterio.open(path) as src:
                # Read all bands at once
                bands_data = src.read([i + 1 for i in band_indices])
                data_cube[:,:,:, idt] = bands_data.transpose(1, 2, 0)
        else:
            print(f"Missing file: {path}, filling with nodata")
            data_cube[:,:,:, idt] = dummy_value
                
    return data_cube

In [ ]:

class FastInferencePipeline:
    def __init__(self, model, scaler, device, batch_size=131072):
        self.model = model.to(device)
        self.scaler = scaler
        self.device = device
        self.batch_size = batch_size
        self.model.eval()
    
    def normalize_data(self, data):
        """Efficiently normalize data using vectorized operations"""
        # data shape is (pixels, n_features, window_size)
        original_shape = data.shape
        
        # (pixels, n_features, window_size) -> (pixels, window_size, n_features) -> (pixels * window_size, n_features)
        # Reshape to (pixels * window_size, n_features) for scaling --> normalization for each channel
        # -> when flattened, consecutive rows represent consecutive time steps for the same pixel
        data_2d = data.transpose(0, 2, 1).reshape(-1, original_shape[1])
        
        # Create mask for valid data
        valid_mask = data_2d != -9999
        
        # Process each valid row
        valid_rows = ~np.all(valid_mask == False, axis=1)
        if np.any(valid_rows):
            data_2d[valid_rows] = self.scaler.transform(data_2d[valid_rows])
        
        data_2d[~valid_rows] = 0

        # numpy clip between 0 and 1
        data_2d = np.clip(data_2d, 0, 1)
        
        # Reshape back to original format: (pixels, n_features, window_size)
        return data_2d.reshape(original_shape[0], original_shape[2], original_shape[1]).transpose(0, 2, 1)
    
    # no need of gradient computation
    @torch.no_grad()
    def __call__(self, data_cube):
        # Reshape and prepare data
        # data_cube has shape (height, width, n_features, window_size)

        # calculation total number of pixels, eg pixels = 5000*5000 = 25.000.000
        pixels = data_cube.shape[0] * data_cube.shape[1]

        # reshape to (25.000.000, 6, 8) --> each row ONE pixel with its 6 features across 8 time steps
        input_data = data_cube.reshape(pixels, data_cube.shape[2], data_cube.shape[3])
        
        # Print shape before normalization
        print(f"Shape before normalization: {input_data.shape}")
        
        # Normalize
        input_data = self.normalize_data(input_data)
        
        # Print shape after normalization
        print(f"Shape after normalization: {input_data.shape}")
        
        # Convert to tensor
        input_tensor = torch.from_numpy(input_data).float()
        
        # Process in batches
        predictions = []
        n_batches = (len(input_tensor) + self.batch_size - 1) // self.batch_size
        
        for i in tqdm(range(0, len(input_tensor), self.batch_size), 
                     total=n_batches, desc="Processing batches"):
            batch = input_tensor[i:i+self.batch_size].to(self.device)
            pred = torch.sigmoid(self.model(batch)) > 0.5
            predictions.append(pred.cpu())
        
        # Combine predictions
        return torch.cat(predictions, dim=0).numpy()

In [ ]:
def save_predictions(predictions, output_file, meta, chunk_size=1024):
    """Save predictions efficiently using chunked writing"""
    # predictions shape: (height, width, bands)
    # rasterio expects: (bands, height, width)
    predictions_transposed = predictions.transpose(2, 0, 1)
    
    with rasterio.open(output_file, 'w', **meta) as dst:
        # Write all bands at once
        dst.write(predictions_transposed)

In [3]:
def main():
    # Load model and scaler
    print("Loading model and scaler...")
    model = AlbasUNet()
    model.load_state_dict(torch.load(model_path))
    scaler = joblib.load(scaler_path)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Get metadata from first BAP file
    bap_files = sorted([f for f in os.listdir(dir_path) if '_BAP.tif' in f])
    if not bap_files:
        print("No BAP files found!")
        return
        
    print(f"Found {len(bap_files)} BAP files")
    
    # Get metadata from first file
    with rasterio.open(os.path.join(dir_path, bap_files[0])) as src:
        height, width = src.height, src.width
        crs = src.crs
        transform = src.transform
    
    # Initialize pipeline
    pipeline = FastInferencePipeline(model, scaler, device, batch_size)
    
    # Process each year
    years_to_process = list(range(1984 + window_size - 1, 2024))
    print(f"Processing years: {years_to_process[0]} to {years_to_process[-1]}")
    
    for target_year in tqdm(years_to_process, desc="Processing years"):
        print(f"\nProcessing year {target_year}")
        
        # Setup output file
        output_pred_file = os.path.join(
            output_dir, 
            f"{target_year}_disturbed_undisturbed_pred_unet_lansatbands_w8.tif"
        )
        
        # Get file paths for the time window
        file_paths = [
            os.path.join(dir_path, f"{year}0801_LEVEL3_LNDLG_BAP.tif")
            for year in range(target_year - window_size + 1, target_year + 1)
        ]
        
        # Load and preprocess data
        data_cube = load_and_preprocess_data(
            file_paths,
            band_indices,
            (height, width, n_features, window_size)
        )
        
        # Process with pipeline
        predictions = pipeline(data_cube)
        
        # Reshape predictions
        predictions_raster = predictions.reshape(height, width, window_size)
        predictions_raster = predictions_raster.astype(np.uint8)
        
        # Setup metadata for saving
        meta = {
            'driver': 'GTiff',
            'height': height,
            'width': width,
            'count': window_size,
            'dtype': rasterio.uint8,
            'crs': crs,
            'transform': transform,
            'compress': 'lzw',
            'tiled': True,
            'blockxsize': 256,
            'blockysize': 256,
            'interleave': 'band'
        }
        
        # Save predictions
        try:
            save_predictions(predictions_raster, output_pred_file, meta)
            print(f"Successfully saved predictions for year {target_year}")
        except Exception as e:
            print(f"Error saving predictions for year {target_year}: {e}")
            
        # Clear GPU memory
        torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

/tmp/ipykernel_29280/1121359961.py:127: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Using device: cuda
Loading model and scaler...
Found 40 BAP files
Processing years: 1991 to 2023


Processing years:   0%|          | 0/33 [00:00<?, ?it/s]


Processing year 1991


Loading files: 100%|██████████| 8/8 [00:17<00:00,  2.20s/it]


...

Successfully saved predictions for year 2022

Processing year 2023


Loading files: 100%|██████████| 8/8 [00:19<00:00,  2.39s/it]


Shape before normalization: (25000000, 6, 8)
Shape after normalization: (25000000, 6, 8)


Processing years: 100%|██████████| 33/33 [25:50<00:00, 47.00s/it]

Successfully saved predictions for year 2023
